# Event Study explicat pas cu pas

Acest notebook porneste direct din logica din `event-study/main.py`, `model.py` si `t_test.py`.

Scopul este sa construim treptat intuitia matematica:

- cum sunt adunate datele
- ce inseamna preturi vs. randamente
- cum arata modelul folosit in proiect
- de ce acest model poate fi vazut ca o regresie liniara foarte simpla
- cum se fac predictiile
- ce inseamna `t-test` si `p-value`

Important: in codul actual nu avem o regresie liniara cu mai multe variabile explicative. Avem varianta cea mai simpla posibil, adica un model doar cu intercept.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.float_format", lambda x: f"{x:.6f}")


def find_sample_path():
    candidates = [
        Path("playin-around-with-yahoo-finance-just-testin/yahoo-finance-outputs/history.jsonc"),
        Path("../playin-around-with-yahoo-finance-just-testin/yahoo-finance-outputs/history.jsonc"),
        Path("../../playin-around-with-yahoo-finance-just-testin/yahoo-finance-outputs/history.jsonc"),
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError("Nu am gasit fisierul history.jsonc cu esantionul local de date.")


## 1. Cum sunt adunate datele

In proiect, ideea centrala este foarte simpla: se descarca seria de preturi de inchidere (`Close`) pentru un activ financiar.

Versiunea scurta din cod este:

```python
import yfinance as yf

data = yf.download("TLV.RO", start="2005-01-01", end="2026-01-01")["Close"]
data = data.squeeze()
```

Ca notebook-ul sa ruleze usor si fara dependenta de internet, mai jos incarcam un esantion deja salvat local in repo.


In [ ]:
sample_path = find_sample_path()

with sample_path.open("r", encoding="utf-8") as f:
    payload = json.load(f)

history_df = pd.DataFrame(payload["data"], columns=payload["columns"])
history_df.index = pd.to_datetime(payload["index"], utc=True).tz_convert(None)
history_df.index.name = "Date"

history_df.head(8)


Observatie:

- `Open`, `High`, `Low`, `Close` sunt preturile din ziua respectiva
- `Volume` este volumul tranzactionat
- pentru modelul din proiect ne intereseaza in primul rand coloana `Close`


## 2. Preturi vs. randamente

Pretul este valoarea efectiva a activului intr-o zi, de exemplu `38.10` lei.

Randamentul spune cu cat s-a miscat relativ pretul fata de ziua anterioara:

$$
R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1
$$

Unde:

- $P_t$ este pretul de azi
- $P_{t-1}$ este pretul de ieri
- $R_t$ este randamentul procentual

In finante se lucreaza foarte des cu randamente, nu cu preturi brute, pentru ca randamentele sunt mai usor de comparat intre active si intre perioade.


In [ ]:
prices = history_df["Close"].rename("Pret inchidere")
returns = prices.pct_change().rename("Randament")

pd.concat([prices, returns], axis=1).head(10)


### Diferenta financiara dintre "actual values" si "randament"

- `actual values` sau valorile efective inseamna, in practica, preturile observate in piata
- `real_returns` din cod inseamna randamentele calculate din acele preturi
- `normal_returns` inseamna ce spune modelul ca ar fi fost o miscare normala
- `abnormal_returns` este diferenta dintre ce s-a intamplat si ce ar fi fost normal

Exemplu:

- daca pretul urca de la `36` la `38`, valoarea efectiva s-a schimbat cu `2` lei
- dar randamentul este `2 / 36 = 0.0556`, adica aproximativ `5.56%`

Randamentul raspunde la intrebarea financiara mai utila: "cu cat la suta s-a miscat activul?".


## 3. Fereastra de estimare si fereastra de eveniment

In `main.py`, codul separa perioada in doua bucati:

- o **fereastra de estimare**: zile linistite din trecut, folosite pentru a invata ce inseamna comportament normal
- o **fereastra de eveniment**: zilele din jurul evenimentului, unde verificam daca miscarea a fost neobisnuita

Pentru un exemplu scurt si usor de urmarit, folosim mai jos un esantion mic din datele locale.


In [ ]:
historical_prices = prices.iloc[:13]
event_prices = prices.iloc[13:19]

historical_returns = historical_prices.pct_change().dropna()
real_returns = event_prices.pct_change().dropna()

rezumat_ferestre = pd.DataFrame(
    {
        "Start": [historical_prices.index.min(), event_prices.index.min()],
        "Stop": [historical_prices.index.max(), event_prices.index.max()],
        "Numar preturi": [len(historical_prices), len(event_prices)],
        "Numar randamente": [len(historical_returns), len(real_returns)],
    },
    index=["Fereastra de estimare", "Fereastra de eveniment"],
)

rezumat_ferestre


## 4. Modelul din proiect ca regresie liniara simpla

Codul din `model.py` face doua lucruri:

```python
self.historical_returns = historical_data.pct_change().dropna()
self.expected_daily_return = self.historical_returns.mean()
```

Asta inseamna ca modelul spune:

$$
r_t = \alpha + \varepsilon_t
$$

Unde:

- $r_t$ este randamentul din ziua $t$
- $\alpha$ este randamentul mediu "normal"
- $\varepsilon_t$ este zgomotul aleator

Acesta este, de fapt, un model de regresie liniara doar cu intercept.

Antrenarea inseamna estimarea lui $\alpha$. Daca folosim metoda celor mai mici patrate, obtinem:

$$
\hat{\alpha} = \arg\min_{\alpha} \sum_{t=1}^{n}(r_t - \alpha)^2 = \bar{r}
$$

Adica parametrul optim este pur si simplu media randamentelor istorice.


In [ ]:
exemplu_randamente = pd.Series([0.0100, -0.0050, 0.0150], name="r_t")
alpha_exemplu = exemplu_randamente.mean()

print(f"Randamente exemplu: {list(exemplu_randamente)}")
print(f"Alpha estimat = media = {alpha_exemplu:.6f} ({alpha_exemplu * 100:.3f}%)")


In [ ]:
expected_daily_return = historical_returns.mean()
historical_std = historical_returns.std()

rezumat_model = pd.Series(
    {
        "Randament mediu zilnic estimat": expected_daily_return,
        "Randament mediu zilnic estimat (%)": expected_daily_return * 100,
        "Volatilitate istorica": historical_std,
        "Volatilitate istorica (%)": historical_std * 100,
    }
)

rezumat_model


## 5. Cum se face predictia

In `model.py`, predictia este:

```python
return np.repeat(self.expected_daily_return, event_window_length)
```

Adica pentru fiecare zi din fereastra de eveniment, modelul spune acelasi lucru:

$$
\hat{r}_t = \hat{\alpha}
$$

Acest model nu incearca sa anticipeze zig-zag-ul zilnic. El spune doar: "daca nu era nimic special, m-as astepta la un randament zilnic apropiat de media istorica".


In [ ]:
normal_returns = np.repeat(expected_daily_return, len(real_returns))
abnormal_returns = real_returns.to_numpy() - normal_returns

comparatie = pd.DataFrame(
    {
        "Randament real": real_returns.to_numpy(),
        "Randament normal prezis": normal_returns,
        "Randament anormal": abnormal_returns,
    },
    index=real_returns.index,
)

comparatie.mul(100).rename(columns=lambda c: f"{c} (%)").round(3)


Formula-cheie pentru partea de Event Study este:

$$
AR_t = r_t - \hat{r}_t
$$

Unde:

- $AR_t$ = randament anormal
- $r_t$ = randamentul real observat
- $\hat{r}_t$ = randamentul normal prezis de model

Daca $AR_t$ este aproape de zero, ziua respectiva seamana cu o zi normala.
Daca $AR_t$ este mare in valoare absoluta, miscarea pare iesita din comun.


## 6. `t-test` si `p-value`

Codul din `t_test.py` calculeaza mai intai media randamentelor anormale:

$$
\overline{AR} = \frac{1}{n}\sum_{t=1}^{n} AR_t
$$

Apoi calculeaza statistica t:

$$
t = \frac{\overline{AR}}{s / \sqrt{n}}
$$

Unde:

- $\overline{AR}$ este media randamentelor anormale
- $s$ este abaterea standard istorica a randamentelor
- $n$ este numarul de zile din fereastra de eveniment

Apoi se calculeaza `p-value` pentru un test bilateral:

$$
p = 2 \cdot \left(1 - F_t(|t|; n-1)\right)
$$

Interpretare intuitiva:

- un `p-value` mic inseamna ca ar fi rar sa vezi o abatere atat de mare doar din zgomot
- in cod, pragul este `0.05`
- daca `p < 0.05`, rezultatul este tratat ca semnificativ statistic


In [ ]:
mean_ar = np.mean(abnormal_returns)
n = len(abnormal_returns)
t_stat = mean_ar / (historical_std / np.sqrt(n))
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n - 1))
is_significant = p_value < 0.05

rezultat_test = pd.Series(
    {
        "Media randamentelor anormale": mean_ar,
        "Media randamentelor anormale (%)": mean_ar * 100,
        "Numar observatii": n,
        "t-statistic": t_stat,
        "p-value": p_value,
        "Semnificativ la prag de 5%": is_significant,
    }
)

rezultat_test


## 7. Ce trebuie retinut

Logica proiectului este aceasta:

1. iei preturile de inchidere
2. le transformi in randamente
3. inveti randamentul zilnic normal dintr-o perioada istorica
4. prezici randamentul normal in fereastra de eveniment
5. calculezi randamente anormale
6. verifici cu un `t-test` daca abaterea pare reala sau doar zgomot

Pe scurt:

- preturile spun "cat valoreaza activul"
- randamentele spun "cu cat s-a miscat"
- modelul spune "care ar fi fost miscarea normala"
- `p-value` spune daca abaterea observata pare suficient de neobisnuita incat sa o luam in serios statistic
